# AirBnB Database Disaster



In [ ]:
!pip -q install duckdb pandas

In [ ]:
import duckdb, pandas as pd, os

HANDOUT = "Lecture1_Airbnb_Handout_45x22.csv"

if os.path.exists(HANDOUT):
    handout_path = HANDOUT
else:
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(
            f"Put {HANDOUT} next to this notebook, or run this in Colab and upload it."
        )
    uploaded = files.upload()
    handout_path = next(iter(uploaded))

print("using:", handout_path)
con = duckdb.connect(database=":memory:")

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE airbnb AS
SELECT
  *,
    TRY_CAST(REPLACE(REGEXP_REPLACE(price, '[^0-9.,]', ''), ',', '') AS DOUBLE) AS price_num,
  TRY_CAST(review_scores_rating AS DOUBLE)                 AS rating_num,
  TRY_CAST(minimum_nights AS INTEGER)                      AS min_nights_num,
  TRY_CAST(availability_365 AS INTEGER)                    AS avail_num,
  TRY_CAST(host_since AS DATE)                             AS host_since_date
FROM read_csv_auto(?, header=True, nullstr=['', 'NA', 'NaN']);
""", [handout_path])


con.sql("SELECT * FROM airbnb LIMIT 5").df()

### Q1. How many distinct hosts are in the excerpt?
*Count unique `host_id` values (safer than names).*

In [ ]:

con.execute("""
SELECT
  COUNT(DISTINCT host_id)   AS distinct_hosts_by_id,
  COUNT(DISTINCT host_name) AS distinct_hosts_by_name
FROM airbnb;
""").df()


### Q2. Which area has the highest average price?
*Group by `neighbourhood_cleansed`, average the numeric `price`.*

In [ ]:
con.execute("""
SELECT neighbourhood_cleansed AS area,
       ROUND(AVG(price_num), 2) AS avg_price
FROM airbnb
GROUP BY area
ORDER BY avg_price DESC;
""").df()

### Q3. Which neighbourhood has the highest share of missing prices?

*Neighbourhood with the highest % of missing prices*

In [ ]:

con.execute("""
WITH per_area AS (
  SELECT
    neighbourhood_cleansed AS area,
    COUNT(*) AS total_rows,
    SUM(CASE WHEN price_num IS NULL THEN 1 ELSE 0 END) AS missing_price_rows
  FROM airbnb
  GROUP BY area
)
SELECT
  area,
  missing_price_rows,
  total_rows,
  ROUND(100.0 * missing_price_rows / NULLIF(total_rows,0), 1) AS missing_pct
FROM per_area
ORDER BY missing_pct DESC, total_rows DESC, area
LIMIT 5;

""").df()


### Q4. For listings with room type “Entire home/apt”, what is the median of `minimum_nights` in the excerpt?
*Filter to that room type and take the median.*

In [ ]:

con.execute("""
SELECT
  MIN(min_nights_num) FILTER (WHERE room_type = 'Entire home/apt') AS min_entire,
  QUANTILE_DISC(min_nights_num, 0.5) FILTER (WHERE room_type = 'Entire home/apt') AS median_entire_disc,
  COUNT(*) FILTER (WHERE room_type = 'Entire home/apt' AND min_nights_num IS NOT NULL) AS n_used
FROM airbnb;
""").df()


### Q5. What is the cheapest neighbourhood on average?
*Group by `neighbourhood_cleansed` and pick the lowest average price.*

In [ ]:
con.execute("""
SELECT neighbourhood_cleansed AS area,
       ROUND(AVG(price_num), 2) AS avg_price
FROM airbnb
GROUP BY area
ORDER BY avg_price ASC;
""").df()

### Q5 again. What if a missing price means zero?

Same data, same question, one decision changed. `AVG` skips a missing price;
`COALESCE(price_num, 0)` counts it as free. Read the two columns side by side
and watch the cheapest area change.

In [ ]:
con.execute("""
SELECT neighbourhood_cleansed AS area,
       ROUND(AVG(price_num), 2)              AS avg_priced_only,
       ROUND(AVG(COALESCE(price_num, 0)), 2) AS avg_missing_as_zero,
       COUNT(*)                              AS n_rows,
       COUNT(price_num)                      AS n_priced
FROM airbnb
GROUP BY area
ORDER BY avg_priced_only ASC;
""").df()

### Is nineteen rows a fair test?

Optional, and only if someone asks. This runs the same average over the full
Amsterdam file: 10,168 listings, about 23 MB. Upload `listings.csv` when prompted,
or let the cell skip itself.

Two things to point at. First, how long it takes. Second, `max_price`: a handful of
listings priced at 50,000 and 80,018 euros are enough to put Buitenveldert-Zuidas and
IJburg at the top. Ranking by `MEDIAN(price_num)` instead puts De Pijp first. The
machine is fast, and it still needs you to decide what the question means.

In [ ]:
import os, time
from IPython.display import display

FULL = "listings.csv"
if not os.path.exists(FULL):
    try:
        from google.colab import files
        FULL = next(iter(files.upload()))
    except Exception:
        FULL = None

if FULL is None:
    print("listings.csv is not here. Skipping.")
else:
    t0 = time.perf_counter()
    out = con.execute("""
    SELECT neighbourhood_cleansed AS area,
           ROUND(AVG(TRY_CAST(REPLACE(REGEXP_REPLACE(price, '[^0-9.,]', ''), ',', '') AS DOUBLE)), 2) AS avg_price,
           MAX(TRY_CAST(REPLACE(REGEXP_REPLACE(price, '[^0-9.,]', ''), ',', '') AS DOUBLE))          AS max_price,
           COUNT(*) AS n_rows
    FROM read_csv_auto(?, header=True, nullstr=['', 'NA', 'NaN'])
    GROUP BY area
    ORDER BY avg_price DESC
    LIMIT 5;
    """, [FULL]).df()
    print("%.2f seconds over the whole file" % (time.perf_counter() - t0))
    display(out)